In [5]:
# ============================================================
#   E-Commerce Data Analysis Project
#   Step 3 — Feature Extraction
#   File   : feature_extraction.py
#   Input  : cleaned_superstore.csv
#   Output : featured_superstore.csv
# ============================================================

import pandas as pd

# ============================================================
# STEP 1 — Load the Cleaned Dataset
# ============================================================

df = pd.read_csv("cleaned_superstore.csv")

# Convert date columns back to real dates after loading CSV
df['Order Date'] = pd.to_datetime(df['Order Date'])
df['Ship Date']  = pd.to_datetime(df['Ship Date'])

print("=" * 55)
print("STEP 1 — Cleaned Dataset Loaded")
print("=" * 55)
print(f"Rows    : {df.shape[0]}")
print(f"Columns : {df.shape[1]}")


# ============================================================
# STEP 2 — Date Based Features
# ============================================================
# Why? Dates alone don't tell us much.
# But from dates we can extract season, week, day — very useful for EDA.

print("\n" + "=" * 55)
print("STEP 2 — Extracting Date Based Features")
print("=" * 55)

# --- Feature 1: Season ---
# Which season was the order placed in?
# Helps answer: "Which season has most sales?"

def get_season(month):
    if month in [12, 1, 2]:   return 'Winter'
    elif month in [3, 4, 5]:  return 'Spring'
    elif month in [6, 7, 8]:  return 'Summer'
    else:                      return 'Fall'

df['Season'] = df['Order Month'].apply(get_season)
print(f"✅ Feature 1 — Season added")
print(df['Season'].value_counts().to_string())


# --- Feature 2: Order Week Number ---
# Which week of the year was the order placed?
# Week 1 = first week of January, Week 52 = last week of December

df['Order Week'] = df['Order Date'].dt.isocalendar().week.astype(int)
print(f"\n✅ Feature 2 — Order Week added (1 to 52)")


# --- Feature 3: Day of the Week ---
# Was the order placed on Monday, Tuesday... Sunday?
# Helps answer: "Which day of the week has most orders?"

df['Order DayOfWeek'] = df['Order Date'].dt.day_name()
print(f"✅ Feature 3 — Order DayOfWeek added")
print(df['Order DayOfWeek'].value_counts().to_string())


# --- Feature 4: Is Weekend ---
# Was the order placed on Saturday or Sunday?
# True = Weekend order, False = Weekday order

df['Is Weekend'] = df['Order Date'].dt.dayofweek >= 5
# dt.dayofweek: Monday=0, Tuesday=1 ... Saturday=5, Sunday=6
# So >= 5 means Saturday or Sunday

print(f"\n✅ Feature 4 — Is Weekend added")
print(df['Is Weekend'].value_counts().to_string())


# ============================================================
# STEP 3 — Sales Based Features
# ============================================================
# Why? Raw sales numbers are hard to compare.
# We create categories and flags to group orders easily.

print("\n" + "=" * 55)
print("STEP 3 — Extracting Sales Based Features")
print("=" * 55)

# --- Feature 5: Sales Category (Bins) ---
# Group each order into Low / Medium / High / Very High
# Based on the Sales amount

df['Sales Category'] = pd.cut(
    df['Sales'],
    bins   = [0, 50, 200, 1000, 25000],
    labels = ['Low', 'Medium', 'High', 'Very High']
)
# pd.cut() divides the Sales into 4 buckets:
# 0-50      → Low
# 50-200    → Medium
# 200-1000  → High
# 1000+     → Very High

print(f"✅ Feature 5 — Sales Category added")
print(df['Sales Category'].value_counts().to_string())


# --- Feature 6: Is High Value Order ---
# Is this order above the average sales amount?
# True = above average, False = below average

average_sales = df['Sales'].mean()
df['Is High Value'] = df['Sales'] > average_sales

print(f"\n✅ Feature 6 — Is High Value added  (Average Sales = ${average_sales:.2f})")
print(df['Is High Value'].value_counts().to_string())


# ============================================================
# STEP 4 — Shipping Based Features
# ============================================================
# Why? "Days to Ship" is a number. But grouping into labels
# makes it easy to understand and compare in EDA.

print("\n" + "=" * 55)
print("STEP 4 — Extracting Shipping Based Features")
print("=" * 55)

# --- Feature 7: Shipping Speed Label ---
# Convert number of days into a speed label

def get_shipping_speed(days):
    if days <= 1:   return 'Express'   # 0 or 1 day
    elif days <= 3: return 'Fast'      # 2 to 3 days
    elif days <= 5: return 'Normal'    # 4 to 5 days
    else:           return 'Slow'      # 6+ days

df['Shipping Speed'] = df['Days to Ship'].apply(get_shipping_speed)

print(f"✅ Feature 7 — Shipping Speed added")
print(df['Shipping Speed'].value_counts().to_string())


# ============================================================
# STEP 5 — Customer Based Features
# ============================================================
# Why? We want to identify important and loyal customers.
# These features help find top customers for EDA.

print("\n" + "=" * 55)
print("STEP 5 — Extracting Customer Based Features")
print("=" * 55)

# --- Feature 8: Customer Order Count ---
# How many total orders has this customer placed?
# transform() assigns the group result back to every row of that customer

df['Customer Order Count'] = df.groupby('Customer Name')['Order ID'].transform('count')

print(f"✅ Feature 8 — Customer Order Count added")
print(f"   Max orders by one customer : {df['Customer Order Count'].max()}")
print(f"   Min orders by one customer : {df['Customer Order Count'].min()}")


# --- Feature 9: Customer Total Sales ---
# What is the total money spent by this customer across all orders?

df['Customer Total Sales'] = df.groupby('Customer Name')['Sales'].transform('sum')

print(f"\n✅ Feature 9 — Customer Total Sales added")
print(f"   Highest customer spend : ${df['Customer Total Sales'].max():,.2f}")
print(f"   Lowest  customer spend : ${df['Customer Total Sales'].min():,.2f}")


# --- Feature 10: Customer Value Segment ---
# Classify each customer as High / Medium / Low value
# Based on their total spending

def customer_value(total_sales):
    if total_sales >= 10000:  return 'High Value'
    elif total_sales >= 3000: return 'Medium Value'
    else:                     return 'Low Value'

df['Customer Value'] = df['Customer Total Sales'].apply(customer_value)

print(f"\n✅ Feature 10 — Customer Value added")
print(df['Customer Value'].value_counts().to_string())


# ============================================================
# STEP 6 — Product Based Features
# ============================================================
# Why? We want to know which products are popular and which are not.

print("\n" + "=" * 55)
print("STEP 6 — Extracting Product Based Features")
print("=" * 55)

# --- Feature 11: Product Order Count ---
# How many times has this product been ordered total?

df['Product Order Count'] = df.groupby('Product Name')['Order ID'].transform('count')

print(f"✅ Feature 11 — Product Order Count added")
print(f"   Most ordered product count  : {df['Product Order Count'].max()}")
print(f"   Least ordered product count : {df['Product Order Count'].min()}")


# --- Feature 12: Product Popularity ---
# Label each product as Popular / Moderate / Rare

def product_popularity(count):
    if count >= 10:  return 'Popular'
    elif count >= 4: return 'Moderate'
    else:            return 'Rare'

df['Product Popularity'] = df['Product Order Count'].apply(product_popularity)

print(f"\n✅ Feature 12 — Product Popularity added")
print(df['Product Popularity'].value_counts().to_string())


# ============================================================
# STEP 7 — Final Summary
# ============================================================

print("\n" + "=" * 55)
print("STEP 7 — Feature Extraction Summary")
print("=" * 55)

new_features = [
    'Season', 'Order Week', 'Order DayOfWeek', 'Is Weekend',
    'Sales Category', 'Is High Value',
    'Shipping Speed',
    'Customer Order Count', 'Customer Total Sales', 'Customer Value',
    'Product Order Count', 'Product Popularity'
]

print(f"Original Columns : 21")
print(f"New Features     : {len(new_features)}")
print(f"Total Columns Now: {df.shape[1]}")
print(f"\nNew Features Added:")
for i, feat in enumerate(new_features, 1):
    print(f"  {i:02d}. {feat}")


# ============================================================
# STEP 8 — Save the Featured Dataset
# ============================================================

df.to_csv("featured_superstore.csv", index=False)

print("\n" + "=" * 55)
print("✅ Feature Extraction Completed Successfully!")
print("✅ File saved as : featured_superstore.csv")
print("=" * 55)

STEP 1 — Cleaned Dataset Loaded
Rows    : 4922
Columns : 21

STEP 2 — Extracting Date Based Features
✅ Feature 1 — Season added
Season
Fall      1828
Spring    1046
Winter    1028
Summer    1020

✅ Feature 2 — Order Week added (1 to 52)
✅ Feature 3 — Order DayOfWeek added
Order DayOfWeek
Tuesday      926
Saturday     898
Sunday       850
Monday       799
Wednesday    635
Friday       552
Thursday     262

✅ Feature 4 — Is Weekend added
Is Weekend
False    3174
True     1748

STEP 3 — Extracting Sales Based Features
✅ Feature 5 — Sales Category added
Sales Category
Low          2438
Medium       1250
High         1014
Very High     220

✅ Feature 6 — Is High Value added  (Average Sales = $220.01)
Is High Value
False    3767
True     1155

STEP 4 — Extracting Shipping Based Features
✅ Feature 7 — Shipping Speed added
Shipping Speed
Normal     2449
Fast       1158
Slow        887
Express     428

STEP 5 — Extracting Customer Based Features
✅ Feature 8 — Customer Order Count added
   Max o